In [67]:
import pandas as pd

In [ ]:
#Import du df -> adresse du csv à vérifier à récupérer sur votre ordinateur
adresse=r"C:\Users\XXXXXX\data\données_traitées\Bibliographie_de_la_France_Tables_Littérature_Version détaillée_1824_v3.csv" #csv annoté (1824_v3) avec prédictions
df=pd.read_csv(adresse, encoding="utf-8",sep=";",header=0,dtype=str)
df = df.fillna("")
df['Identifiant ouvrage']=df['Identifiant ouvrage'].str.replace("http://ark.bnf.fr/ark:/","https://catalogue.bnf.fr/ark:/")
df["Identifiant auteur"]=df["Identifiant auteur"].str.replace("http://ark.bnf.fr/ark:/","https://catalogue.bnf.fr/ark:/")
df["Identifiant para-auteur"]=df["Identifiant para-auteur"].str.replace("http://ark.bnf.fr/ark:/","https://catalogue.bnf.fr/ark:/")

In [69]:
def verification_ouvrage(row):
    verite=row['Identifiant ouvrage'].split(";")
    prediction=row['Ouvrage_prédit'].split(" , ")
    resultat="Hors BNF"
    for vrai_ouvrage in verite:
        if not vrai_ouvrage.startswith("https://catalogue.bnf.fr/ark:/"):#Il ne s'agit pas d'un lien BNF, l'algorithme n'est normalement pas en capacité de le trouver
            continue 
        elif vrai_ouvrage in prediction:
            resultat="Prediction correcte"
            break #Dans le cas de multiples ouvrages, on se contentera d'un seul correctement prédit
        elif prediction==[""]:
            resultat="Non trouvé"
        else:
            resultat="Pas le même ouvrage"
    return resultat
df['predict_ouvrage_verif']=df.apply(verification_ouvrage,axis=1)
print(df['predict_ouvrage_verif'].value_counts())

predict_ouvrage_verif
Prediction correcte    1304
Non trouvé              384
Hors BNF                232
Pas le même ouvrage     187
Name: count, dtype: int64


In [57]:
#Visualisation direct des résultats d'un type (ouvrage)
verif_ouvrage="Non trouvé" #En choisir un parmi "Prediction correcte", "Non trouvé", "Hors BNF", "Pas le même ouvrage"
df[df['predict_ouvrage_verif']==verif_ouvrage][["Titre","Auteur","Identifiant ouvrage","Ouvrage_prédit","Auteur_prédit",'predict_ouvrage_verif']]

,Titre,Auteur,Identifiant ouvrage,Ouvrage_prédit,Auteur_prédit,predict_ouvrage_verif
11,Réflexions rapides sur une rébellion de débite...,l'Hermite du quartier Notre-Dame,https://catalogue.bnf.fr/ark:/12148/cb36388010m,,,Non trouvé
16,L’Entrée de Ferdinand VII à Madrid ; stances p...,"J. Brisset, garde du corps du roi de France",https://catalogue.bnf.fr/ark:/12148/cb301627424,,,Non trouvé
21,"Le Tambour de Logrono, ou Jeunesse et Valeur, ...",Adolphe C***;Combault,https://catalogue.bnf.fr/ark:/12148/cb30520618c,,,Non trouvé
22,"L’Arc de triomphe, tableau-vaudeville […], rep...",M. Carmouche;M. Emile Vander Burch,https://catalogue.bnf.fr/ark:/12148/cb301974366,,,Non trouvé
24,"Mon hommage à S. A. R. Mgr le duc d’Angoulême,...",Capelle,https://catalogue.bnf.fr/ark:/12148/cb30193716f,,,Non trouvé
...,...,...,...,...,...,...
2085,Petit Télémaque ou précis des avantures de Tél...,un instituteur,https://catalogue.bnf.fr/ark:/12148/cb33533867g,,,Non trouvé
2095,Fables nouvelles. Tome II,Auguste Rigaud,https://catalogue.bnf.fr/ark:/12148/cb31217142z,,,Non trouvé
2100,Pamphlet des pamphlets. Deuxième édition,"Paul Louis Courier, vigneron",https://catalogue.bnf.fr/ark:/12148/cb36315982s,,,Non trouvé
2102,"La Pauvre enfant, chant élégiaque",,https://catalogue.bnf.fr/ark:/12148/cb335286370,,,Non trouvé


In [64]:
def verification_auteur(row):
    auteur_True,paraauteur_True,auteur_predict,paraauteur_predict=row["Identifiant auteur"],row["Identifiant para-auteur"],row["Auteur_prédit"],row["Autre_auteur_prédit"]
    auteur_True=auteur_True.split(";") if auteur_True else []
    auteur_True=set([code for code in auteur_True if "https://catalogue.bnf.fr/" in code])
    paraauteur_True=paraauteur_True.split(";") if paraauteur_True else []
    paraauteur_True=set([code for code in paraauteur_True if "https://catalogue.bnf.fr/" in code])
    auteur_predict=auteur_predict.split(";") if auteur_predict else []
    auteur_predict=set([name[name.find("http"):-2] for name in auteur_predict])
    paraauteur_predict=paraauteur_predict.split(";") if paraauteur_predict else []
    paraauteur_predict=set([name[name.find("http"):-2] for name in paraauteur_predict])
    results=dict()
    if auteur_True==auteur_predict:
        if auteur_True:
            results["Auteur"]="ok!"
        else:
            results["Auteur"]="ok(vide)"
    else:
        result={"Auteur présent":0,"Auteur placé en para-auteur":0,"Auteur absent":0,"Auteur ajouté":0}
        for auteur in auteur_True:
            if auteur in auteur_predict:
                result["Auteur présent"]+=1
            elif auteur in paraauteur_predict:
                result["Auteur placé en para-auteur"]+=1
            else:
                result["Auteur absent"]+=1
        for auteur in auteur_predict:
            if auteur not in auteur_True and auteur not in paraauteur_True:
                result["Auteur ajouté"]+=1
        listtemp=[]
        for k,v in result.items():
            if v>0:
                listtemp.append(k)
        if not listtemp:
            listtemp=["ok(vide)"]
        elif listtemp==["Auteur présent"]:
            listtemp=["ok!"]
        results['Auteur']=" ; ".join(listtemp)

    if paraauteur_True==paraauteur_predict:
        if paraauteur_True:
            results["Para-auteur"]="ok!"
        else:
            results["Para-auteur"]="ok(vide)"
    else:
        result={"Para-auteur présent":0,"Para-auteur placé en auteur":0,"Para-auteur absent":0,"Para-auteur ajouté":0}
        for paraauteur in paraauteur_True:
            if paraauteur in paraauteur_predict:
                result["Para-auteur présent"]+=1
            elif paraauteur in auteur_predict:
                result["Para-auteur placé en auteur"]+=1
            else:
                result["Para-auteur absent"]+=1
        for paraauteur in paraauteur_predict:
            if paraauteur not in paraauteur_True:
                result["Para-auteur ajouté"]+=1
        listtemp=[]
        for k,v in result.items():
            if v>0:
                listtemp.append(k)
        if not listtemp:
            listtemp=["ok(vide)"]
        elif listtemp==["Para-auteur présent"]:
            listtemp=["ok!"]
        results['Para-auteur']=" ; ".join(listtemp)
    return pd.Series(results)
df[['predict_auteur_verif','predict_paraauteur_verif']]=df.apply(verification_auteur,axis=1,result_type="expand")

#Modification des résultats pours quelques auteurs correctement identifiés mais dont les données d'entraînement donne un mauvais code (un code plus valable ?)
df.loc[(df["Auteur"] == "J. J. Rousseau") & (df["Auteur_prédit"] == "Rousseau, Jean-Jacques (1712-1778) [https://catalogue.bnf.fr/ark:/12148/cb119228797]"), "auteur_verification"] = "ok"
df.loc[(df["Auteur"] == "Gresset") & (df["Auteur_prédit"] == "Gresset, Jean-Baptiste-Louis (1709-1777) [https://catalogue.bnf.fr/ark:/12148/cb12063085m]"), "auteur_verification"] = "ok"
df.loc[(df["Auteur"] == "J. Racine") & (df["Auteur_prédit"] == "Racine, Jean (1639-1699) [https://catalogue.bnf.fr/ark:/12148/cb120076761]"), "auteur_verification"] = "ok"
df.loc[(df["Auteur"] == "Ovide") & (df["Auteur_prédit"] == "Ovide (0043 av. J.-C.-0017) [https://catalogue.bnf.fr/ark:/12148/cb119183166]"), "auteur_verification"] = "ok"

print(f"--- Sur tous les textes : {len(df)} textes")
print(df['predict_auteur_verif'].value_counts())
print(" ")
print(df['predict_paraauteur_verif'].value_counts())
print(" ")
verif_ouvrage="Prediction correcte"
print(f"--- Sur les textes correctement identifiés : {len(df[df['predict_ouvrage_verif']==verif_ouvrage])} textes")
print(df[df['predict_ouvrage_verif']==verif_ouvrage]['predict_auteur_verif'].value_counts())
print(" ")
print(df[df['predict_ouvrage_verif']==verif_ouvrage]['predict_paraauteur_verif'].value_counts())
print(" ")
verif_ouvrage="Hors BNF"
print(f"--- Sur les textes hors BNF (potentiellement identifiés quand même) : {len(df[df['predict_ouvrage_verif']==verif_ouvrage])} textes")
print(df[df['predict_ouvrage_verif']==verif_ouvrage]['predict_auteur_verif'].value_counts())
print(" ")
print(df[df['predict_ouvrage_verif']==verif_ouvrage]['predict_paraauteur_verif'].value_counts())
print(" ")
verif_ouvrage="Pas le même ouvrage"
print(f"--- Sur les textes liés à un autre de la BNF que celui prévu : {len(df[df['predict_ouvrage_verif']==verif_ouvrage])} textes")
print(df[df['predict_ouvrage_verif']==verif_ouvrage]['predict_auteur_verif'].value_counts())
print(" ")
print(df[df['predict_ouvrage_verif']==verif_ouvrage]['predict_paraauteur_verif'].value_counts())

--- Sur tous les textes
predict_auteur_verif
ok!                                               1091
Auteur absent                                      358
ok(vide)                                           306
Auteur ajouté                                      145
Auteur absent ; Auteur ajouté                      101
Auteur présent ; Auteur ajouté                      52
Auteur placé en para-auteur                         26
Auteur présent ; Auteur absent                      13
Auteur présent ; Auteur absent ; Auteur ajouté       7
Auteur placé en para-auteur ; Auteur ajouté          3
Auteur placé en para-auteur ; Auteur absent          3
Auteur présent ; Auteur placé en para-auteur         2
Name: count, dtype: int64
 
predict_paraauteur_verif
ok(vide)                                                                                       1653
Para-auteur ajouté                                                                              207
Para-auteur absent                        

In [66]:
#Visualisation direct des résultats d'un type (auteur)
verif_ouvrage="Hors BNF" #En choisir un parmi "Prediction correcte", "Non trouvé", "Hors BNF", "Pas le même ouvrage"
verif_auteur="ok!"  #En choisir un parmi "ok!","ok(vide)","Auteur ajouté","Auteur absent","Auteur présent","Auteur placé en para-auteur"
df[(df['predict_ouvrage_verif']==verif_ouvrage) & (df['predict_auteur_verif'].str.contains(verif_auteur))][["Titre","Auteur","Identifiant auteur","Identifiant para-auteur","Auteur_prédit","Autre_auteur_prédit",'predict_auteur_verif','predict_paraauteur_verif']]

,Titre,Auteur,Identifiant auteur,Identifiant para-auteur,Auteur_prédit,Autre_auteur_prédit,predict_auteur_verif,predict_paraauteur_verif
0,"Classiques français. Poésies de Malherbe, plus...",Malherbe,https://catalogue.bnf.fr/ark:/12148/cb12327103c,,"Malherbe, François de (1555-1628) [ https://ca...",,ok!,ok(vide)
13,Nouvelles Méditations poétiques. Troisième édi...,Alphonse de Lamartine,https://catalogue.bnf.fr/ark:/12148/cb119108008,,"Lamartine, Alphonse de (1790-1869) [ https://c...",Thibaron (18..-1885) [ https://catalogue.bnf.f...,ok!,Para-auteur ajouté
37,"Ecole des vieillards, comédie en cinq actes et...",Casimir Delavigne,https://catalogue.bnf.fr/ark:/12148/cb12052639n,,"Delavigne, Casimir (1793-1843) [ https://catal...",,ok!,ok(vide)
53,Elémens de la grammaire latine,Lhomond,https://catalogue.bnf.fr/ark:/12148/cb12170155m,,"Lhomond, Charles-François (1727-1794) [ https:...",,ok!,ok(vide)
81,Répertoire du Théâtre-Français. Œuvres complèt...,Crébillon,https://catalogue.bnf.fr/ark:/12148/cb11998713x,,"Crébillon, Prosper de (1674-1762) [ https://ca...",,ok!,ok(vide)
91,"Cadet Buteux à l’Ecole des vieillards, pot-pou...","M. Jacinthe Leclere, convive des Soupers de Momus",https://catalogue.bnf.fr/ark:/12148/cb17748957j,,"Leclère, Jacinthe [ https://catalogue.bnf.fr/a...",,ok!,ok(vide)
104,The Works of lord Byron,lord Byron,https://catalogue.bnf.fr/ark:/12148/cb11894686q,,"Byron, George Gordon (1788-1824) [ https://cat...","Lake, J. W. (17..?-18..) [ https://catalogue.b...",ok!,Para-auteur ajouté
244,"Elémens de la grammaire française, par C. F. L...","C. F. Lhomond, professeur émérite en la ci-dev...",https://catalogue.bnf.fr/ark:/12148/cb12170155m,,"Lhomond, Charles-François (1727-1794) [ https:...",,ok!,ok(vide)
245,"Grammaire française de Lhomond, à l'usage des ...",Lhomond,https://catalogue.bnf.fr/ark:/12148/cb12170155m,https://catalogue.bnf.fr/ark:/12148/cb13005487s,"Lhomond, Charles-François (1727-1794) [ https:...",,ok!,Para-auteur absent
258,"Phædri, liberti Augusti fabulæ. Nova editio st...","Phèdre, affranchi d'Auguste",https://catalogue.bnf.fr/ark:/12148/cb11985800x,,Phèdre (0015? av. J.-C.-0054) [ https://catalo...,,ok!,ok(vide)


In [ ]:
#Si vous êtes plus à l'aise sur logiciel : vous pouvez le re-sauvegarder avec les colonnes de verification ainsi
adresse="" #adresse de sauvegarde
df.to_csv(adresse, encoding="utf-8", sep=";", header=True, index=False)